<a href="https://colab.research.google.com/github/billurbektas/TP_JSDM/blob/main/Calanda_JSDM/R/02_model/05_sjsdm_experiments_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# sjSDM Model Experiments Pipeline

**Experiments:**
1. Full grid search (spatial_form x alpha x lambda)
2. Decoupled lambda sensitivity
3. Drop-one environmental variable
4. k-fold CV (species AUC, train-test gap, site log-loss)
5. ANOVA sampling saturation

Each run extracts: overall R², full model E/S/C, species-level R²/E/S/C, site-level R²/E/S/C

## Setup: Install packages & mount Drive

The Google drive mounting needs to be conducted when the runtime is configured for Python 3.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


From here on, the runtime needs to be changed to R.

In [1]:
install.packages("sjSDM")
install.packages("tidyverse")
install.packages("conflicted")
install.packages("pROC")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘foreach’, ‘iterators’, ‘RcppTOML’, ‘here’, ‘png’, ‘plyr’, ‘doParallel’, ‘gridExtra’, ‘reticulate’, ‘mvtnorm’, ‘abind’, ‘Metrics’, ‘checkmate’, ‘mathjaxr’, ‘beeswarm’, ‘qgam’, ‘viridis’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [2]:
library(sjSDM)
install_sjSDM(version = "gpu")


── Attaching sjSDM ──────────────────────────────────────────────────── 1.0.7 ──

✖ torch 
✖ torch_optimizer 
✖ pyro 
✖ madgrad 


Torch or other dependencies not found:

	1. Use install_sjSDM() to install Pytorch and conda automatically 
	2. Installation trouble shooting guide: ?installation_help 
	3. If 1) and 2) did not help, please create an issue on <https://github.com/TheoreticalEcology/s-jSDM/issues> (see ?install_diagnostic) 

* Installing Miniconda -- please wait a moment ...

* Downloading 'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh' ...

+ /usr/bin/bash /tmp/Rtmp9GJJwW/Miniforge3-Linux-x86_64.sh -b -u -p '/root/.local/share/r-miniconda'

+ /root/.local/share/r-miniconda/bin/conda update --yes --name base conda

+ /root/.local/share/r-miniconda/bin/conda create --yes --name r-reticulate 'python=3.12' numpy --quiet -c conda-forge

* Miniconda has been successfully installed at "~/.local/share/r-miniconda".

+ /root/.local/share

In [3]:
# Verify GPU
system("nvidia-smi", intern = TRUE)
system("/root/.local/share/r-miniconda/bin/conda run -n r-sjsdm pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --force-reinstall")

[1] "Mon Mar  2 10:34:11 2026       "                                                            
 [2] "+-----------------------------------------------------------------------------------------+"
 [3] "| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |"
 [4] "+-----------------------------------------+------------------------+----------------------+"
 [5] "| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |"
 [6] "| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |"
 [7] "|                                         |                        |               MIG M. |"
 [8] "|=========================================+========================+======================|"
 [9] "|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |"
[10] "| N/A   32C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |"
[11] "|                                         |                        |             Disabled |"
[12] "+-----------------------------------------+------------------------+----------------------+"
[13] ""                                                                                           
[14] "+-----------------------------------------------------------------------------------------+"
[15] "| Processes:                                                                              |"
[16] "|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |"
[17] "|        ID   ID                                                               Usage      |"
[18] "|=========================================================================================|"
[19] "|  No running processes found                                                             |"
[20] "+-----------------------------------------------------------------------------------------+"

Here, the runtime needs to be restarted. On the above menu: **Run all --> Restart session**

In [1]:
library(reticulate)
use_condaenv("r-sjsdm", required = TRUE)

torch <- import("torch")
cat("PyTorch version:", torch$`__version__`, "\n")
cat("CUDA available:", torch$cuda$is_available(), "\n")
cat("CUDA version:", torch$version$cuda, "\n")

PyTorch version: 2.5.1+cu121 
CUDA available: TRUE 
CUDA version: 12.1 


## Load libraries & data

In [2]:
library(sjSDM)
library(tidyverse)
library(conflicted)
library(pROC)

conflict_prefer("select", "dplyr")
conflict_prefer("filter", "dplyr")
conflict_prefer("intersect", "base")

set.seed(42)
seed = 42
session_info = sessionInfo()

── Attaching sjSDM ──────────────────────────────────────────────────── 1.0.7 ──

✔ torch <environment> 
✔ torch_optimizer  
✔ pyro  
✔ madgrad  


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Type 'citation("pROC")' for a citation.

[conflicted] Will prefer dplyr::select over any other package.
[conflicted] Will prefer dplyr::filter over any other package.
[conflicted] Will prefer base::intersect over any other package.


In [3]:
# ============================================================================
# USER SETTINGS
# ============================================================================

# Google Drive paths
drive_base = "/content/drive/MyDrive/A_ETH/Calanda_JSDM"
data_file = file.path(drive_base, "data_calanda_jsdm_2026-02-25.rds")
results_dir = file.path(drive_base, "results")

# Model settings
device = "gpu"
iterations = 700L
learning_rate = 0.01
sampling_fit = 1000L
anova_samples_fast = 1000L

# Environmental formula
env_formula = as.formula("~summer_temp + fdd + et.annual + slope + rocks_cover +
  trees_cover + shrubs_cover + soil_depth_var +
  tpi + flowdir + nutrient + disturbance")
print(env_formula)

# Toggle experiments
run_exp1 = TRUE   # grid search
run_exp2 = TRUE   # decoupled lambdas
run_exp3 = TRUE   # drop-one predictor
run_exp4 = TRUE   # k-fold CV
run_exp5 = TRUE   # ANOVA saturation

# Representative config for Exp 3-5
rep_spatial = "LIN_XY_XY" # updated after the Exp 1-2
rep_alpha = 1 # updated after the Exp 1-2
rep_lambda = 0.01 # updated after the Exp 1-2
rep_lambda_env = 0.001 # updated after the Exp 2

~summer_temp + fdd + et.annual + slope + rocks_cover + trees_cover + 
    shrubs_cover + soil_depth_var + tpi + flowdir + nutrient + 
    disturbance


In [4]:
# Load data
cat("Loading data from:", data_file, "\n")
data_calanda_jsdm = readRDS(data_file)
X = data_calanda_jsdm$X
Y = data_calanda_jsdm$Y

cat("X:", nrow(X), "sites x", ncol(X), "predictors\n")
cat("Y:", nrow(Y), "sites x", ncol(Y), "species\n")
cat("X columns:", paste(colnames(X), collapse = ", "), "\n")

altitude = X[, "altitude"]
XY = X[, c("Latitude", "Longitude")]
richness = rowSums(Y)
env_cols = all.vars(env_formula)
Xenv = X[, env_cols, drop = FALSE]

# Create results directories on Drive
for (d in c("results", "results/runs", "results/decoupled",
            "results/dropone", "results/cv", "results/saturation_fit")) {
  dir.create(file.path(drive_base, d), showWarnings = FALSE, recursive = TRUE)
}

Loading data from: /content/drive/MyDrive/A_ETH/Calanda_JSDM/data_calanda_jsdm_2026-02-25.rds 
X: 576 sites x 18 predictors
Y: 576 sites x 352 species
X columns: Longitude, Latitude, altitude, slope, summer_temp, fdd, et.annual, soil_depth_mean, soil_depth_var, trees_cover, shrubs_cover, rocks_cover, flowdir, tpi, roughness, snow_sum, nutrient, disturbance 


## Core functions

In [5]:
fit_sjsdm = function(Y_train, Xenv_train, XY_train,
                       env_form = env_formula,
                       spatial_form = "DNN",
                       lambda_env = 0.1, alpha_env = 0.5,
                       lambda_sp = 0.1, alpha_sp = 0.5,
                       lambda_bio = 0.1, alpha_bio = 0.5) {

    env_component = do.call(linear, list(
      data = Xenv_train,
      formula = env_form,
      lambda = lambda_env,
      alpha = alpha_env
    ))

    if (spatial_form == "DNN") {
      sp_component = DNN(
        XY_train,
        formula = ~0 + .,
        hidden = c(30L, 30L),
        activation = "relu",
        bias = FALSE,
        lambda = lambda_sp, alpha = alpha_sp
      )
    } else if (spatial_form == "LIN_XY_XY") {
      sp_component = do.call(linear, list(
        data = XY_train,
        formula = ~0 + Latitude + Longitude + I(Latitude * Longitude),
        lambda = lambda_sp,
        alpha = alpha_sp
      ))
    } else {
      stop("Unknown spatial_form: ", spatial_form)
    }

    sjSDM(
      Y = Y_train,
      env = env_component,
      spatial = sp_component,
      biotic = bioticStruct(lambda = lambda_bio, alpha = alpha_bio,
                            df = ncol(Y_train), reg_on_Cov = FALSE),
      iter = iterations,
      device = device,
      learning_rate = learning_rate,
      sampling = sampling_fit,
      control = sjSDMControl(
        RMSprop(weight_decay = 0.0),
        scheduler = 5L,
        early_stopping_training = 25L,
        lr_reduce_factor = 0.9
      ),
      se = FALSE
    )
  }

In [6]:
extract_partition = function(model, anova_samples = anova_samples_fast) {

    R2 = Rsquared(model, verbose = FALSE)
    an = anova(model, verbose = FALSE, samples = anova_samples)
    res = internalStructure(an, fractions = "proportional",
                            Rsquared = "McFadden", plot = FALSE)

    # --- Species-level (r2 already in output) ---
    sp = res$internals$Species

    # --- Site-level ---
    si = res$internals$Sites

    # --- Summary ---
    summary_row = tibble(
      # Model-level
      overall_R2 = R2,
      # Species
      species_R2_mean = mean(sp$r2, na.rm = TRUE),
      species_E_mean = mean(sp$env, na.rm = TRUE),
      species_S_mean = mean(sp$spa, na.rm = TRUE),
      species_C_mean = mean(sp$codist, na.rm = TRUE),
      species_R2_median = median(sp$r2, na.rm = TRUE),
      species_E_median = median(sp$env, na.rm = TRUE),
      species_S_median = median(sp$spa, na.rm = TRUE),
      species_C_median = median(sp$codist, na.rm = TRUE),
      # Sites
      site_R2_mean = mean(si$r2, na.rm = TRUE),
      site_E_mean = mean(si$env, na.rm = TRUE),
      site_S_mean = mean(si$spa, na.rm = TRUE),
      site_C_mean = mean(si$codist, na.rm = TRUE),
      site_R2_median = median(si$r2, na.rm = TRUE),
      site_E_median = median(si$env, na.rm = TRUE),
      site_S_median = median(si$spa, na.rm = TRUE),
      site_C_median = median(si$codist, na.rm = TRUE)
    )

    list(
      R2 = R2,
      anova = an,
      internal_structure = res,
      species = sp,
      sites = si,
      summary = summary_row
    )
  }


In [7]:
build_stratified_folds = function(k = 10, alt, tree_cover, shrub_cover,
                                    disturbance, nutrient, richness,
                                    site_names = NULL,
                                    woody_thresh = 0, # because the values are z-score standardized this means that we are taking above average values.
                                    seednum = seed,
                                    plot = TRUE) {
    n = length(alt)

    set.seed(seednum)

    elev_bin = as.integer(cut(alt,
      breaks = quantile(alt, probs = seq(0, 1, length.out = 6)),
      include.lowest = TRUE))

    woody_class = ifelse((tree_cover + shrub_cover) > woody_thresh, 1L, 0L)

    landuse_score = scale(disturbance)[, 1] + scale(nutrient)[, 1]
    landuse_bin = as.integer(cut(landuse_score,
      breaks = quantile(landuse_score, probs = seq(0, 1, length.out = 4), na.rm = TRUE),
      include.lowest = TRUE))
    landuse_bin[is.na(landuse_bin)] = 1L

    richness_bin = as.integer(cut(richness,
      breaks = quantile(richness, probs = seq(0, 1, length.out = 4)),
      include.lowest = TRUE))

    strata = paste(elev_bin, woody_class, landuse_bin, richness_bin, sep = "_")
    strata_factor = as.factor(strata)
    fold_ids = rep(NA_integer_, n) #- Creates an empty vector of fold IDs for all n sites
    if (!is.null(site_names)) names(fold_ids) = site_names


  # - For each stratum:
  #  - idx = which sites belong to this stratum (e.g. 7 sites)
  #  - perm = sample(seq_len(k)) = random permutation of 1-10, e.g. [6, 3, 9, 1, 7, 4, 10, 2, 8, 5]
  #  - rep(perm, length.out = 7) = [6, 3, 9, 1, 7, 4, 10] — assigns the 7 sites to folds in that random order, recycling if the stratum had more than 10 sites

    for (s in levels(strata_factor)) {
      idx = which(strata_factor == s)
      perm = sample(seq_len(k))
      fold_ids[idx] = rep(perm, length.out = length(idx))
    }

    if (plot) {
      cat("\n--- Counts per elevation bin per fold ---\n")
      print(table(fold = fold_ids, elev_bin = elev_bin))

      cat("\n--- Counts per land-use bin per fold ---\n")
      print(table(fold = fold_ids, landuse_bin = landuse_bin))

      cat("\n--- Counts per richness bin per fold ---\n")
      print(table(fold = fold_ids, richness_bin = richness_bin))

      cat("\n--- Counts per woody class per fold ---\n")
      print(table(fold = fold_ids, woody_class = woody_class))

      cat("\n--- Min/max altitude and woody fraction per fold ---\n")
      fold_diag = tibble(fold = fold_ids, altitude = alt, woody = woody_class) %>%
        group_by(fold) %>%
        summarise(
          n = n(),
          alt_min = min(altitude),
          alt_max = max(altitude),
          woody_frac = mean(woody),
          .groups = "drop"
        )
      print(as.data.frame(fold_diag))

      # --- PCA plot ---
      df_strata = tibble(
        altitude = scale(alt)[, 1],
        woody_cover = scale(tree_cover + shrub_cover)[, 1],
        landuse_score = scale(landuse_score)[, 1],
        richness = scale(richness)[, 1],
        fold = factor(fold_ids),
        stratum = strata_factor
      )

      pca_strata = prcomp(df_strata %>% select(altitude, woody_cover, landuse_score, richness))
      df_pca = tibble(
        PC1 = pca_strata$x[, 1],
        PC2 = pca_strata$x[, 2],
        fold = df_strata$fold
      )

      var_exp = round(100 * summary(pca_strata)$importance[2, 1:2], 1)

      loadings = as.data.frame(pca_strata$rotation[, 1:2])
      loadings$variable = rownames(loadings)
      arrow_scale = max(abs(df_pca$PC1), abs(df_pca$PC2)) * 0.8

      p = ggplot(df_pca, aes(x = PC1, y = PC2, color = fold)) +
        geom_point(alpha = 0.6, size = 2) +
        geom_segment(data = loadings,
                     aes(x = 0, y = 0, xend = PC1 * arrow_scale, yend = PC2 * arrow_scale),
                     inherit.aes = FALSE,
                     arrow = arrow(length = unit(0.2, "cm")),
                     color = "grey30", linewidth = 0.6) +
        geom_text(data = loadings,
                  aes(x = PC1 * arrow_scale * 1.12, y = PC2 * arrow_scale * 1.12, label = variable),
                  inherit.aes = FALSE, size = 3.2, color = "grey20") +
        labs(title = "Stratified folds in strata-variable PCA space",
             x = paste0("PC1 (", var_exp[1], "%)"),
             y = paste0("PC2 (", var_exp[2], "%)"),
             color = "Fold") +
        theme_bw() +
        theme(legend.position = "right")

      print(p)
    }

    folds = lapply(seq_len(k), function(i) {
      idx = which(fold_ids == i)
      if (!is.null(site_names)) names(idx) = site_names[idx]
      idx
    })
    folds
  }


In [ ]:
# Check the balance of the design
k = 10
    cat("Config:", rep_spatial, "alpha=", rep_alpha, "lambda=", rep_lambda, "\n")

    folds = build_stratified_folds(
      k = k, alt = altitude,
      tree_cover = Xenv[, "trees_cover"],
      shrub_cover = Xenv[, "shrubs_cover"],
      disturbance = Xenv[, "disturbance"],
      nutrient = Xenv[, "nutrient"],
      richness = richness,
      site_names = rownames(Y)
    )
    str(folds)
    cat("Fold sizes:", paste(sapply(folds, length), collapse = ", "), "\n\n")

## Experiment 1: Full grid search

In [ ]:
if (run_exp1) {
  cat("\n", strrep("=", 60), "\n")
    cat("EXPERIMENT 1: Full grid search\n")
    cat(strrep("=", 60), "\n")

    grid = expand.grid(
      spatial_form = c("DNN", "LIN_XY_XY"),
      alpha_common = c(0, 0.5, 1),
      lambda_common = c(0.001, 0.01, 0.1, 0.5, 1, 2),
      stringsAsFactors = FALSE
    ) %>%
      mutate(run_id = paste0(spatial_form, "_a", alpha_common, "_l", lambda_common))

    cat("Total runs:", nrow(grid), "\n\n")

    print(grid) # check the grid design
  }

In [ ]:
if (run_exp1) {
  exp1_summaries = list()

    for (i in seq_len(nrow(grid))) {

      cfg = grid[i, ]
      run_file = file.path(drive_base, "results", "runs", paste0(cfg$run_id, ".rds"))

      if (file.exists(run_file)) {
        cat("[", i, "/", nrow(grid), "] Cached:", cfg$run_id, "\n")
        flush.console()
        run_data = readRDS(run_file)
      } else {
        cat("[", i, "/", nrow(grid), "] Fitting:", cfg$run_id, "\n")
        flush.console()

        t0 = Sys.time()
        model = fit_sjsdm(
          Y_train = Y, Xenv_train = Xenv, XY_train = XY,
          spatial_form = cfg$spatial_form,
          env_form = env_formula,
          lambda_env = cfg$lambda_common, alpha_env = cfg$alpha_common,
          lambda_sp = cfg$lambda_common, alpha_sp = cfg$alpha_common,
          lambda_bio = cfg$lambda_common, alpha_bio = cfg$alpha_common
        )
        t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  Fit done in", t_fit, "min\n")
        flush.console()

        cat("[", i, "/", nrow(grid), "] Variance partitioning:", cfg$run_id, "\n")
        flush.console()
        t0 = Sys.time()
        partition = extract_partition(model)
        t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  VP done in", t_vp, "min\n")
        flush.console()

        run_data = list(config = cfg, model = model, partition = partition)
        saveRDS(run_data, run_file)
        cat("  -> Saved (total:", as.numeric(t_fit) + as.numeric(t_vp), "min)\n")
        flush.console()
      }

      exp1_summaries[[i]] = bind_cols(cfg, run_data$partition$summary)
    }

    summary_exp1 = bind_rows(exp1_summaries)
    write_csv(summary_exp1, file.path(drive_base, "results", "summary_exp1_grid.csv"))
    cat("\nSaved summary_exp1_grid.csv\n")
    flush.console()
    print(summary_exp1)
  }

## Experiment 2: Decoupled lambda sensitivity

In [ ]:
if (run_exp2) {

  cat("\n", strrep("=", 60), "\n")
  cat("EXPERIMENT 2: Decoupled lambda sensitivity\n")
  cat(strrep("=", 60), "\n")

  alpha_fixed = 0.5
  lambda_anchor = 0.1
  lambda_values = c(0.01, 0.1, 0.5, 1, 2)

  grid_decoupled = bind_rows(
    expand.grid(spatial_form = c("DNN", "LIN_XY_XY"),
                vary = "lambda_env", lambda_vary = lambda_values,
                stringsAsFactors = FALSE),
    expand.grid(spatial_form = c("DNN", "LIN_XY_XY"),
                vary = "lambda_sp", lambda_vary = lambda_values,
                stringsAsFactors = FALSE),
    expand.grid(spatial_form = c("DNN", "LIN_XY_XY"),
                vary = "lambda_bio", lambda_vary = lambda_values,
                stringsAsFactors = FALSE)
  ) %>%
    mutate(
      lambda_env = ifelse(vary == "lambda_env", lambda_vary, lambda_anchor),
      lambda_sp = ifelse(vary == "lambda_sp", lambda_vary, lambda_anchor),
      lambda_bio = ifelse(vary == "lambda_bio", lambda_vary, lambda_anchor),
      run_id = paste0("decoupled_", spatial_form, "_", vary, "_", lambda_vary)
    )

  cat("Total runs:", nrow(grid_decoupled), "\n\n")

  exp2_summaries = list()

  for (i in seq_len(nrow(grid_decoupled))) {

    cfg = grid_decoupled[i, ]
    run_file = file.path(drive_base, "results", "decoupled", paste0(cfg$run_id, ".rds"))

    if (file.exists(run_file)) {
      cat("[", i, "/", nrow(grid_decoupled), "] Cached:", cfg$run_id, "\n")
      run_data = readRDS(run_file)
    } else {
      cat("[", i, "/", nrow(grid_decoupled), "] Fitting:", cfg$run_id, "\n")

      model = fit_sjsdm(
        Y_train = Y, Xenv_train = Xenv, XY_train = XY,
        spatial_form = cfg$spatial_form,
        lambda_env = cfg$lambda_env, alpha_env = alpha_fixed,
        lambda_sp = cfg$lambda_sp, alpha_sp = alpha_fixed,
        lambda_bio = cfg$lambda_bio, alpha_bio = alpha_fixed
      )

      partition = extract_partition(model)

      run_data = list(config = cfg, model = model, partition = partition)
      saveRDS(run_data, run_file)
      cat("  -> Saved\n")
    }

    exp2_summaries[[i]] = bind_cols(
      cfg %>% select(run_id, spatial_form, vary, lambda_vary, lambda_env, lambda_sp, lambda_bio),
      run_data$partition$summary
    )
  }

  summary_exp2 = bind_rows(exp2_summaries)
  write_csv(summary_exp2, file.path(drive_base, "results", "summary_exp2_decoupled.csv"))
  cat("\nSaved summary_exp2_decoupled.csv\n")
  print(summary_exp2)
}


## Experiment 2b: Reproducibility check for two identical runs with same configuration



In [ ]:
 cat("\n", strrep("=", 60), "\n")
  cat("REPRODUCIBILITY CHECK: Two identical LIN_XY_XY a=1 l=0.01 fits\n")
  cat(strrep("=", 60), "\n")

  repro_config = list(spatial_form = "LIN_XY_XY", alpha = 1, lambda = 0.01)

  for (run_i in 1:2) {
    run_file = file.path(drive_base, "results", paste0("repro_check_run", run_i, ".rds"))

    if (file.exists(run_file)) {
      cat("[Run", run_i, "] Cached\n")
      flush.console()
    } else {
      cat("[Run", run_i, "] Fitting...\n")
      flush.console()

      t0 = Sys.time()
      model = fit_sjsdm(
        Y_train = Y, Xenv_train = Xenv, XY_train = XY,
        spatial_form = repro_config$spatial_form,
        env_form = env_formula,
        lambda_env = repro_config$lambda, alpha_env = repro_config$alpha,
        lambda_sp  = repro_config$lambda, alpha_sp  = repro_config$alpha,
        lambda_bio = repro_config$lambda, alpha_bio = repro_config$alpha
      )
      t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
      cat("  Fit done in", t_fit, "min\n")
      flush.console()

      cat("[Run", run_i, "] Variance partitioning...\n")
      flush.console()
      t0 = Sys.time()
      partition = extract_partition(model)
      t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
      cat("  VP done in", t_vp, "min\n")
      flush.console()

      run_data = list(config = repro_config, model = model, partition = partition)
      saveRDS(run_data, run_file)
      cat("  -> Saved (total:", as.numeric(t_fit) + as.numeric(t_vp), "min)\n")
      flush.console()
    }
  }

  # --- Compare the two runs ---
  cat("\n", strrep("-", 60), "\n")
  cat("COMPARISON\n")
  cat(strrep("-", 60), "\n")

  r1 = readRDS(file.path(drive_base, "results", "repro_check_run1.rds"))
  r2 = readRDS(file.path(drive_base, "results", "repro_check_run2.rds"))

  # Overall R2
  cat(sprintf("\nOverall R2:  Run1=%.6f  Run2=%.6f  diff=%.6f\n",
              r1$partition$R2, r2$partition$R2,
              r2$partition$R2 - r1$partition$R2))

  # Species-level correlations
  cat("\nSpecies-level correlations (Run1 vs Run2):\n")
  for (col in c("env", "spa", "codist", "r2")) {
    r = cor(r1$partition$species[, col], r2$partition$species[, col], use = "complete.obs")
    cat(sprintf("  %-7s r = %.4f\n", col, r))
  }

  # Site-level correlations
  cat("\nSite-level correlations (Run1 vs Run2):\n")
  for (col in c("env", "spa", "codist", "r2")) {
    r = cor(r1$partition$sites[, col], r2$partition$sites[, col], use = "complete.obs")
    cat(sprintf("  %-7s r = %.4f\n", col, r))
  }

  cat("\n(If reproducible: diffs ~ 0, correlations ~ 1)\n")

## Experiment 3: Drop-one environmental variable

In [8]:
if (run_exp3) {

    cat("\n", strrep("=", 60), "\n")
    cat("EXPERIMENT 3: Drop-one environmental variable\n")
    cat(strrep("=", 60), "\n")
    cat("Config:", rep_spatial, "alpha=", rep_alpha,
        "lambda_env=", rep_lambda_env,
        "lambda_sp=", rep_lambda, "lambda_bio=", rep_lambda, "\n\n")

    # --- Base model (full, all predictors) ---
    base_file = file.path(drive_base, "results", "dropone", "dropone_BASE.rds")

    if (file.exists(base_file)) {
      cat("[BASE] Cached\n")
      flush.console()
      base_data = readRDS(base_file)
    } else {
      cat("[BASE] Fitting full model...\n")
      flush.console()

      t0 = Sys.time()
      model_base = fit_sjsdm(
        Y_train = Y,
        Xenv_train = Xenv,
        XY_train = XY,
        env_form = env_formula,
        spatial_form = rep_spatial,
        lambda_env = rep_lambda_env, alpha_env = rep_alpha,
        lambda_sp = rep_lambda, alpha_sp = rep_alpha,
        lambda_bio = rep_lambda, alpha_bio = rep_alpha
      )
      t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
      cat("  Fit done in", t_fit, "min\n")
      flush.console()

      cat("[BASE] Variance partitioning...\n")
      flush.console()
      t0 = Sys.time()
      partition_base = extract_partition(model_base)
      t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
      cat("  VP done in", t_vp, "min\n")
      flush.console()

      base_data = list(dropped = "NONE", model = model_base, partition = partition_base)
      saveRDS(base_data, base_file)
      cat("  -> Saved (total:", as.numeric(t_fit) + as.numeric(t_vp), "min)\n")
      flush.console()
    }

    # --- Drop-one loop ---
    exp3_summaries = list()

    # Add base model as first row
    exp3_summaries[[1]] = bind_cols(
      tibble(dropped_variable = "NONE"),
      base_data$partition$summary
    )

    for (j in seq_along(env_cols)) {

      pred = env_cols[j]
      run_id = paste0("dropone_", pred)
      run_file = file.path(drive_base, "results", "dropone", paste0(run_id, ".rds"))

      if (file.exists(run_file)) {
        cat("[", j, "/", length(env_cols), "] Cached:", pred, "\n")
        flush.console()
        run_data = readRDS(run_file)
      } else {
        cat("[", j, "/", length(env_cols), "] Dropping:", pred, "\n")
        flush.console()

        t0 = Sys.time()
        reduced_cols = setdiff(env_cols, pred)
        reduced_formula = as.formula(paste("~", paste(reduced_cols, collapse = " + ")))

        model = fit_sjsdm(
          Y_train = Y,
          Xenv_train = Xenv[, reduced_cols, drop = FALSE],
          XY_train = XY,
          env_form = reduced_formula,
          spatial_form = rep_spatial,
          lambda_env = rep_lambda_env, alpha_env = rep_alpha,
          lambda_sp = rep_lambda, alpha_sp = rep_alpha,
          lambda_bio = rep_lambda, alpha_bio = rep_alpha
        )
        t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  Fit done in", t_fit, "min\n")
        flush.console()

        cat("[", j, "/", length(env_cols), "] Variance partitioning:", pred, "\n")
        flush.console()
        t0 = Sys.time()
        partition = extract_partition(model)
        t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  VP done in", t_vp, "min\n")
        flush.console()

        run_data = list(dropped = pred, model = model, partition = partition)
        saveRDS(run_data, run_file)
        cat("  -> Saved (total:", as.numeric(t_fit) + as.numeric(t_vp), "min)\n")
        flush.console()
      }

      exp3_summaries[[j + 1]] = bind_cols(
        tibble(dropped_variable = pred),
        run_data$partition$summary
      )
    }

    summary_exp3 = bind_rows(exp3_summaries)
    write_csv(summary_exp3, file.path(drive_base, "results", "summary_exp3_dropone.csv"))
    cat("\nSaved summary_exp3_dropone.csv\n")
    flush.console()
    print(summary_exp3)
  }



EXPERIMENT 3: Drop-one environmental variable
Config: LIN_XY_XY alpha= 1 lambda_env= 0.001 lambda_sp= 0.01 lambda_bio= 0.01 

[BASE] Fitting full model...
  Fit done in 0.7 min
[BASE] Variance partitioning...
[1] 0.4401575
  VP done in 4.6 min
  -> Saved (total: 5.3 min)
[ 1 / 12 ] Dropping: summer_temp 
  Fit done in 0.5 min
[ 1 / 12 ] Variance partitioning: summer_temp 
[1] 0.4180206
  VP done in 4.5 min
  -> Saved (total: 5 min)
[ 2 / 12 ] Dropping: fdd 
  Fit done in 0.6 min
[ 2 / 12 ] Variance partitioning: fdd 
[1] 0.4351013
  VP done in 4.6 min
  -> Saved (total: 5.2 min)
[ 3 / 12 ] Dropping: et.annual 
  Fit done in 0.5 min
[ 3 / 12 ] Variance partitioning: et.annual 
[1] 0.4328223
  VP done in 4.5 min
  -> Saved (total: 5 min)
[ 4 / 12 ] Dropping: slope 
  Fit done in 0.6 min
[ 4 / 12 ] Variance partitioning: slope 
[1] 0.4338819
  VP done in 4.5 min
  -> Saved (total: 5.1 min)
[ 5 / 12 ] Dropping: rocks_cover 
  Fit done in 0.7 min
[ 5 / 12 ] Variance partitioning: rocks_cov

## Experiment 4: k-fold CV (species AUC, train-test gap, site log-loss)

In [ ]:
 if (run_exp4) {

    cat("\n", strrep("=", 60), "\n")
    cat("EXPERIMENT 4: k-fold cross-validation\n")
    cat(strrep("=", 60), "\n")

    k = 10
    cat("Config:", rep_spatial, "alpha=", rep_alpha,
        "lambda_env=", rep_lambda_env,
        "lambda_sp=", rep_lambda, "lambda_bio=", rep_lambda, "\n")

    folds = build_stratified_folds(
      k = k, alt = altitude,
      tree_cover = Xenv[, "trees_cover"],
      shrub_cover = Xenv[, "shrubs_cover"],
      disturbance = Xenv[, "disturbance"],
      nutrient = Xenv[, "nutrient"],
      richness = richness,
      site_names = rownames(Y)
    )
    cat("Fold sizes:", paste(sapply(folds, length), collapse = ", "), "\n\n")

    P_oof = matrix(NA, nrow = nrow(Y), ncol = ncol(Y),
                   dimnames = list(rownames(Y), colnames(Y)))
    train_auc_per_fold = matrix(NA, nrow = k, ncol = ncol(Y))

    for (fold_i in seq_len(k)) {

      fold_file = file.path(drive_base, "results", "cv", paste0("fold_", fold_i, ".rds"))
      test_idx = folds[[fold_i]]
      train_idx = setdiff(seq_len(nrow(Y)), test_idx)

      if (file.exists(fold_file)) {
        cat("[Fold", fold_i, "/", k, "] Cached\n")
        flush.console()
        fold_data = readRDS(fold_file)
      } else {
        cat("[Fold", fold_i, "/", k, "] Fitting...\n")
        flush.console()

        t0 = Sys.time()
        model_fold = fit_sjsdm(
          Y_train = Y[train_idx, ],
          Xenv_train = Xenv[train_idx, , drop = FALSE],
          XY_train = XY[train_idx, , drop = FALSE],
          spatial_form = rep_spatial,
          env_form = env_formula,
          lambda_env = rep_lambda_env, alpha_env = rep_alpha,
          lambda_sp = rep_lambda, alpha_sp = rep_alpha,
          lambda_bio = rep_lambda, alpha_bio = rep_alpha
        )
        t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  Fit done in", t_fit, "min\n")
        flush.console()

        cat("[Fold", fold_i, "/", k, "] Predicting...\n")
        flush.console()

        preds_test = predict(model_fold,
                             newdata = Xenv[test_idx, , drop = FALSE],
                             SP = XY[test_idx, , drop = FALSE])

        preds_train = predict(model_fold,
                              newdata = Xenv[train_idx, , drop = FALSE],
                              SP = XY[train_idx, , drop = FALSE])

        fold_data = list(
          fold = fold_i,
          train_idx = train_idx,
          test_idx = test_idx,
          preds_test = preds_test,
          preds_train = preds_train
        )
        saveRDS(fold_data, fold_file)
        cat("  -> Saved (total:", t_fit, "min)\n")
        flush.console()
      }

      P_oof[fold_data$test_idx, ] = fold_data$preds_test

      # Train AUC per species for this fold
      for (s in seq_len(ncol(Y))) {
        y_tr = Y[fold_data$train_idx, s]
        p_tr = fold_data$preds_train[, s]
        if (sum(y_tr == 1) >= 2 && sum(y_tr == 0) >= 2) {
          train_auc_per_fold[fold_i, s] = tryCatch(
            as.numeric(auc(roc(y_tr, p_tr, quiet = TRUE))),
            error = function(e) NA
          )
        }
      }
    }

    cat("\nAll folds complete. Computing metrics...\n")
    flush.console()

In [ ]:
# --- Species metrics: test AUC + train-test gap ---
species_cv = tibble(
  species = colnames(Y),
  n_presences = as.integer(colSums(Y)),
  test_auc = NA_real_,
  train_auc_mean = NA_real_,
  train_test_gap = NA_real_
)

for (s in seq_len(ncol(Y))) {
  y_true = Y[, s]
  y_pred = P_oof[, s]
  valid = !is.na(y_pred)

  if (sum(y_true[valid] == 1) >= 2 && sum(y_true[valid] == 0) >= 2) {
    species_cv$test_auc[s] = tryCatch(
      as.numeric(auc(roc(y_true[valid], y_pred[valid], quiet = TRUE))),
      error = function(e) NA
    )
  }

  species_cv$train_auc_mean[s] = mean(train_auc_per_fold[, s], na.rm = TRUE)
  species_cv$train_test_gap[s] = species_cv$train_auc_mean[s] - species_cv$test_auc[s]
}

species_cv = species_cv %>% arrange(test_auc)
write_csv(species_cv, file.path(drive_base, "results", "exp4_species_cv.csv"))
cat("Saved exp4_species_cv.csv\n")

cat("\nSpecies test AUC:\n")
print(summary(species_cv$test_auc))
cat("Train-test gap:\n")
print(summary(species_cv$train_test_gap))

In [ ]:
# --- Site metrics: log-loss ---
eps = 1e-7
site_cv = tibble(
  site = rownames(Y),
  richness = as.integer(richness),
  logloss = NA_real_
)

for (i in seq_len(nrow(Y))) {
  y_true = Y[i, ]
  y_pred = pmin(pmax(P_oof[i, ], eps), 1 - eps)
  if (all(!is.na(y_pred))) {
    site_cv$logloss[i] = -mean(y_true * log(y_pred) + (1 - y_true) * log(1 - y_pred))
  }
}

site_cv = site_cv %>% arrange(desc(logloss))
write_csv(site_cv, file.path(drive_base, "results", "exp4_sites_cv.csv"))
cat("Saved exp4_sites_cv.csv\n")

cat("\nSite log-loss:\n")
print(summary(site_cv$logloss))

## Experiment 5: ANOVA sampling saturation

In [ ]:
 if (run_exp5) {

    cat("\n", strrep("=", 60), "\n")
    cat("EXPERIMENT 5: Sampling saturation\n")
    cat(strrep("=", 60), "\n")

    # --- Part A: Fitting sampling (sampling_fit in sjSDM) ---
    cat("\n--- Part A: Model fitting sampling saturation ---\n")
    flush.console()

    fit_sample_sizes = c(100, 1000, 5000, 10000)
    fit_results = list()

    for (ns in fit_sample_sizes) {
      fit_file = file.path(drive_base, "results", , "saturation_fit", paste0("saturation_fit_", ns, ".rds"))

      if (file.exists(fit_file)) {
        cat("Cached: sampling_fit =", ns, "\n")
        flush.console()
        fit_results[[as.character(ns)]] = readRDS(fit_file)
      } else {
        cat("Fitting: sampling_fit =", ns, "...\n")
        flush.console()

        t0 = Sys.time()
        model = sjSDM(
          Y = Y,
          env = do.call(linear, list(
            data = Xenv,
            formula = env_formula,
            lambda = rep_lambda_env, alpha = rep_alpha
          )),
          spatial = do.call(linear, list(
            data = XY,
            formula = ~0 + Latitude + Longitude + I(Latitude * Longitude),
            lambda = rep_lambda, alpha = rep_alpha
          )),
          biotic = bioticStruct(lambda = rep_lambda, alpha = rep_alpha,
                                df = ncol(Y), reg_on_Cov = FALSE),
          iter = iterations,
          device = device,
          learning_rate = learning_rate,
          sampling = as.integer(ns),
          control = sjSDMControl(
            RMSprop(weight_decay = 0.0),
            scheduler = 5L,
            early_stopping_training = 25L,
            lr_reduce_factor = 0.9
          ),
          se = FALSE
        )
        t_fit = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  Fit done in", t_fit, "min\n")
        flush.console()

        cat("  Variance partitioning (anova_samples =", as.integer(ns), ")...\n")
        flush.console()
        t0 = Sys.time()
        partition = extract_partition(model, anova_samples = as.integer(ns))
        t_vp = round(difftime(Sys.time(), t0, units = "mins"), 1)
        cat("  VP done in", t_vp, "min\n")
        flush.console()

        run_data = list(model = model, partition = partition, sampling_fit = ns)
        saveRDS(run_data, fit_file)
        cat("  -> Saved (total:", as.numeric(t_fit) + as.numeric(t_vp), "min)\n")
        flush.console()

        fit_results[[as.character(ns)]] = run_data
      }
    }

    ref_anova = fit_results[["50000"]]
    sat_anova_table = tibble(anova_samples = fit_sample_sizes)

    for (idx in seq_along(anova_sample_sizes)) {
      p = anova_results[[as.character(anova_sample_sizes[idx])]]

      sat_anova_table$cor_E_site[idx]    = safe_cor(p$sites$env, ref_anova$sites$env)
      sat_anova_table$cor_S_site[idx]    = safe_cor(p$sites$spa, ref_anova$sites$spa)
      sat_anova_table$cor_C_site[idx]    = safe_cor(p$sites$codist, ref_anova$sites$codist)
      sat_anova_table$cor_E_species[idx] = safe_cor(p$species$env, ref_anova$species$env)
      sat_anova_table$cor_S_species[idx] = safe_cor(p$species$spa, ref_anova$species$spa)
      sat_anova_table$cor_C_species[idx] = safe_cor(p$species$codist, ref_anova$species$codist)

      sat_anova_table$mad_E_site[idx]    = safe_mad(p$sites$env, ref_anova$sites$env)
      sat_anova_table$mad_S_site[idx]    = safe_mad(p$sites$spa, ref_anova$sites$spa)
      sat_anova_table$mad_C_site[idx]    = safe_mad(p$sites$codist, ref_anova$sites$codist)
      sat_anova_table$mad_E_species[idx] = safe_mad(p$species$env, ref_anova$species$env)
      sat_anova_table$mad_S_species[idx] = safe_mad(p$species$spa, ref_anova$species$spa)
      sat_anova_table$mad_C_species[idx] = safe_mad(p$species$codist, ref_anova$species$codist)
    }

    write_csv(sat_anova_table, file.path(drive_base, "results", "exp5_saturation_anova.csv"))
    cat("\nSaved exp5_saturation_anova.csv\n")
    flush.console()
    print(sat_anova_table)

    # --- Plots ---

    # Part A plot
    p_fit = sat_fit_table %>%
      select(sampling_fit, starts_with("cor_")) %>%
      pivot_longer(-sampling_fit, names_to = "component", values_to = "correlation") %>%
      ggplot(aes(x = sampling_fit, y = correlation, color = component)) +
      geom_hline(yintercept = 0, linewidth = 0.3, color = "grey40") +
      geom_hline(yintercept = 0.99, linetype = "dashed", color = "grey50") +
      geom_line(linewidth = 0.8) +
      geom_point(size = 2) +
      scale_x_log10() +
      labs(title = "Part A: Fitting sampling saturation (vs 10k reference)",
           subtitle = "Different sampling_fit values in sjSDM(), anova fixed at 1000",
           x = "sampling_fit", y = "Correlation with reference") +
      theme_bw() +
      theme(legend.position = "bottom")

    # Part B plot
    p_anova = sat_anova_table %>%
      select(anova_samples, starts_with("cor_")) %>%
      pivot_longer(-anova_samples, names_to = "component", values_to = "correlation") %>%
      ggplot(aes(x = anova_samples, y = correlation, color = component)) +
      geom_hline(yintercept = 0, linewidth = 0.3, color = "grey40") +
      geom_hline(yintercept = 0.99, linetype = "dashed", color = "grey50") +
      geom_line(linewidth = 0.8) +
      geom_point(size = 2) +
      scale_x_log10() +
      labs(title = "Part B: ANOVA sampling saturation (vs 50k reference)",
           subtitle = "Same fitted model, different anova samples",
           x = "ANOVA samples", y = "Correlation with reference") +
      theme_bw() +
      theme(legend.position = "bottom")

    p_combined = p_fit / p_anova +
      plot_annotation(
        title = "Experiment 5: Sampling saturation",
        theme = theme(plot.title = element_text(face = "bold", size = 13))
      )

    ggsave(file.path(drive_base, "results", "plots", "exp5_saturation.pdf"),
           p_combined, height = 10, width = 10)
    cat("Saved exp5_saturation.pdf\n")
    flush.console()
  }


## Session info

In [ ]:
cat("\n=== All experiments complete ===")
print(session_info)
saveRDS(session_info, file.path(drive_base, "results",
                                paste0("session_info_", Sys.Date(), ".rds")))